In [1]:
%pip install scikit-learn xgboost
%pip install \
    --extra-index-url=https://pypi.nvidia.com \
    "cudf-cu13==26.6.*" "dask-cudf-cu13==26.6.*" "cuml-cu13==26.6.*" \
    "cugraph-cu13==26.6.*" "nx-cugraph-cu13==26.6.*" "cuxfilter-cu13==26.6.*" \
    "cucim-cu13==26.6.*" "pylibraft-cu13==26.6.*" "raft-dask-cu13==26.6.*" \
    "cuvs-cu13==26.6.*" "nvforest-cu13==26.6.*" "nx-cugraph-cu13==26.6.*"
%pip install imblearn

  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl (8.9 MB)
     ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
     ---------------------------------------- 0.1/101.7 MB 2.6 MB/s eta 0:00:40
     ---------------------------------------- 0.3/101.7 MB 3.3 MB/s eta 0:00:32
     ---------------------------------------- 1.1/101.7 MB 7.7 MB/s eta 0:00:14
      -------------------------------------- 2.4/101.7 MB 12.7 MB/s eta 0:00:08
     - ------------------------------------- 3.8/101.7 MB 16.0 MB/s eta 0:00:07
     - ------------------------------------- 5.0/101.7 MB 17.6 MB/s eta 0:00:06
     - ------------------------------------- 5.1/101.7 MB 18.1 MB/s eta 0:00:06
     -- ------------------------------------ 7.2/101.7 MB 20.0 MB/s eta 0:00:05
     --- ----------------------------------- 8.0/101.7 MB 19.7 MB/s eta 0:00:05
     --- ----------------------------------- 9.4/101.7 MB 20.6 MB/s eta 0:00:05
     ---- --------------------------------- 10.8/101.7 MB 


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement cudf-cu13==26.6.* (from versions: 0.0.0a0)
ERROR: No matching distribution found for cudf-cu13==26.6.*

[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.svm import SVC
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import numpy as np
random_seed = 2026
np.random.seed(random_seed)

### The Plan
- Split trainval and test grouped by subject_ID (to prevent data leakage); stratify mind wandering
- KFold cross validation using trainval during hyperparameter gridsearch (I will try to prevent data leakage here as well); stratify mind wandering

## Load Data

In [41]:
# Load data
path = "data/preprocessed_oversample/data_channels_2000ms.pkl"
data = pd.read_pickle(path).reset_index(drop=True)
display(data.head())

# Split data
groups = data["subject"]
X = data[[*data.columns][2:]].astype(float)
y = data["is_mw"].astype(int)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=random_seed)


,subject,is_mw,delta_Fp1,delta_AF7,delta_AF3,delta_F1,delta_F3,delta_F5,delta_F7,delta_FT7,...,gamma_CP4,gamma_CP2,gamma_P2,gamma_P4,gamma_P6,gamma_P8,gamma_P10,gamma_PO8,gamma_PO4,gamma_O2
0,s10014,0.0,4.881890e-12,8.046855e-12,4.906490e-12,8.260127e-12,6.499393e-12,3.713863e-12,6.666549e-12,5.597572e-12,...,7.114171e-14,5.840358e-14,8.916105e-14,8.077792e-14,1.734865e-13,5.823846e-13,1.796040e-13,1.380206e-13,9.937098e-14,1.287732e-13
1,s10014,0.0,1.328723e-12,3.878664e-12,3.237936e-12,3.373525e-12,3.409657e-12,4.244336e-12,7.835318e-12,4.420457e-12,...,6.271098e-14,3.728105e-14,5.557023e-14,5.840797e-14,6.840236e-14,1.233782e-13,1.560745e-13,1.083970e-13,8.411790e-14,1.059149e-13
2,s10014,0.0,7.064399e-12,7.504018e-12,4.081724e-12,4.861283e-12,4.661894e-12,5.551102e-12,1.214769e-11,7.968254e-12,...,1.090314e-13,6.900395e-14,1.567664e-13,7.243003e-14,1.081399e-13,2.110315e-13,3.744295e-13,2.735808e-13,1.413225e-13,5.015883e-13
3,s10014,0.0,7.442551e-13,2.571612e-12,8.545336e-13,1.185973e-12,6.535775e-13,1.037566e-12,3.248629e-12,2.962574e-12,...,8.456892e-14,6.591919e-14,6.018063e-14,8.662366e-14,1.181751e-13,2.356774e-13,2.247668e-13,2.256949e-13,7.101234e-14,2.247730e-13
4,s10014,0.0,3.156075e-12,7.335819e-12,2.313594e-12,4.615782e-12,3.692127e-12,3.895819e-12,1.214717e-11,7.533915e-12,...,1.251623e-13,6.695101e-14,9.949693e-14,1.036055e-13,1.187265e-13,1.839784e-13,3.257695e-13,1.906158e-13,9.740538e-14,3.002975e-13


## Training
### Logistic Regression

In [35]:
def train_logistic_regression(X, y, groups, parameters):
    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_seed)

    lr = LogisticRegression(random_state=random_seed)
    clf = GridSearchCV(lr, parameters, cv=cv, scoring="roc_auc") 

    clf.fit(X,y,groups=groups)
    
    return clf

### SVM with radial basis function

In [36]:
def train_svc(X, y, groups, parameters):
    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_seed)

    svc = SVC(probability = True,random_state=random_seed)
    clf = GridSearchCV(svc, parameters, cv=cv, scoring="roc_auc") 

    clf.fit(X,y,groups=groups)
    
    return clf

### XGBoost

In [37]:
def train_xgboost(X, y, groups, parameters):
    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_seed)

    xgb = XGBClassifier(scale_pos_weight=len(y_trainval[y_trainval==0])/len(y_trainval[y_trainval==1]), random_state=random_seed, eval_metric="auc")
    clf = GridSearchCV(xgb, parameters, cv=cv, scoring="roc_auc") 

    clf.fit(X,y,groups=groups)
    
    return clf

### Training

In [38]:
lr_params = { # Why these?
    'C': [1],#, 10, 100], 
    'class_weight': ["balanced"],
    'solver': ["lbfgs", "liblinear"],
    'max_iter': [750],#,1250],
    'warm_start': [True]#, False]
}

svm_params = { # Why these?
    'C': [1],#, 10, 100],
    'gamma': ["auto"]#,"scale", 0.01, 0.1]
}

xgb_params = { # Why these?
    'learning_rate': [0.01],#, 0.1],
    'n_estimators': [100],#, 200],
    'max_depth': [5],#, 10],
    'reg_alpha': [1],#, 5]
}

In [42]:

for fold, (train_idxs, test_idxs) in enumerate(cv.split(X, y, groups)):
    X_trainval, y_trainval = X.iloc[train_idxs].reset_index(drop=True), y[train_idxs].reset_index(drop=True)
    X_test, y_test = X.iloc[test_idxs].reset_index(drop=True), y[test_idxs].reset_index(drop=True)
    groups_train = groups[train_idxs]
    groups_test = groups[test_idxs]

    print(f"Fold {fold}")
    best_logreg_clf = train_logistic_regression(X_trainval, y_trainval, groups_train, parameters=lr_params)
    perf_auc = roc_auc_score(y_test, best_logreg_clf.predict_proba(X_test)[:,1])
    print(f"ROC-AUC (LogisticRegression({best_logreg_clf.best_params_})): {perf_auc}")

    best_svm_clf = train_svc(X_trainval, y_trainval, groups_train, parameters=svm_params)
    perf_auc = roc_auc_score(y_test, best_svm_clf.decision_function(X_test))
    print(f"ROC-AUC (SVC({best_svm_clf.best_params_})): {perf_auc}")

    best_xgb_clf = train_xgboost(X_trainval, y_trainval, groups_train, parameters=xgb_params)
    perf_auc = roc_auc_score(y_test, best_xgb_clf.predict_proba(X_test)[:,1])
    print(f"ROC-AUC (XGBoost({best_xgb_clf.best_params_})): {perf_auc}")
    # print()
    # print("Fold :", fold)
    # print("TRAIN POSITIVE RATIO:", y[train_idxs].mean())
    # print("TEST POSITIVE RATIO :", y[test_idxs].mean())
    # print(f"TRAIN PERCENTAGE    : {len(train_idxs) / len(X) * 100:.4}%")
    # print("TRAIN GROUPS        :", set(groups[train_idxs]))
    # print("TEST GROUPS         :", set(groups[test_idxs]))


Fold 0
ROC-AUC (LogisticRegression({'C': 1, 'class_weight': 'balanced', 'max_iter': 750, 'solver': 'liblinear', 'warm_start': True})): 0.5
ROC-AUC (SVC({'C': 1, 'gamma': 'auto'})): 0.5
ROC-AUC (XGBoost({'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100, 'reg_alpha': 1})): 0.5341927605354715
Fold 1
ROC-AUC (LogisticRegression({'C': 1, 'class_weight': 'balanced', 'max_iter': 750, 'solver': 'liblinear', 'warm_start': True})): 0.5
ROC-AUC (SVC({'C': 1, 'gamma': 'auto'})): 0.5
ROC-AUC (XGBoost({'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100, 'reg_alpha': 1})): 0.49597461362167244
Fold 2
ROC-AUC (LogisticRegression({'C': 1, 'class_weight': 'balanced', 'max_iter': 750, 'solver': 'liblinear', 'warm_start': True})): 0.5
ROC-AUC (SVC({'C': 1, 'gamma': 'auto'})): 0.5
ROC-AUC (XGBoost({'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100, 'reg_alpha': 1})): 0.43795765172438217
Fold 3
ROC-AUC (LogisticRegression({'C': 1, 'class_weight': 'balanced', 'max_iter': 750, 'solve

In [73]:
groups_train = groups[train_idxs]
groups_train
groups_train = pd.concat([groups_train, pd.Series(np.random.choice(groups_train.unique(), size = rest))])
groups_train

0       s10014
1       s10014
2       s10014
3       s10014
4       s10014
         ...  
4601    s10089
4602    s10014
4603    s10059
4604    s10052
4605    s10081
Length: 12452, dtype: object

In [ ]:
#with smote
for fold, (train_idxs, test_idxs) in enumerate(cv.split(X, y, groups)):
    X_trainval, y_trainval = X.iloc[train_idxs].reset_index(drop=True), y[train_idxs].reset_index(drop=True)
    X_test, y_test = X.iloc[test_idxs].reset_index(drop=True), y[test_idxs].reset_index(drop=True)
    smote = SMOTE(sampling_strategy='minority', random_state=random_seed)
    X_trainval_sm, y_trainval_sm = smote.fit_resample(X_trainval, y_trainval)
    groups_train = groups[train_idxs]
    groups_test = groups[test_idxs]
    rest = len(X_trainval_sm) - len(X_trainval)
    groups_train = pd.concat([groups_train, pd.Series(np.random.choice(groups_train.unique(), size = rest))])

    print(f"Fold {fold}")
    best_logreg_clf = train_logistic_regression(X_trainval_sm, y_trainval_sm, groups_train, parameters=lr_params)
    perf_auc = roc_auc_score(y_test, best_logreg_clf.predict_proba(X_test)[:,1])
    print(f"ROC-AUC (LogisticRegression({best_logreg_clf.best_params_})): {perf_auc}")

    best_svm_clf = train_svc(X_trainval_sm, y_trainval_sm, groups_train, parameters=svm_params)
    perf_auc = roc_auc_score(y_test, best_svm_clf.decision_function(X_test))
    print(f"ROC-AUC (SVC({best_svm_clf.best_params_})): {perf_auc}")

    best_xgb_clf = train_xgboost(X_trainval_sm, y_trainval_sm, groups_train, parameters=xgb_params)
    perf_auc = roc_auc_score(y_test, best_xgb_clf.predict_proba(X_test)[:,1])
    print(f"ROC-AUC (XGBoost({best_xgb_clf.best_params_})): {perf_auc}")
    # print()
    # print("Fold :", fold)
    # print("TRAIN POSITIVE RATIO:", y[train_idxs].mean())
    # print("TEST POSITIVE RATIO :", y[test_idxs].mean())
    # print(f"TRAIN PERCENTAGE    : {len(train_idxs) / len(X) * 100:.4}%")
    # print("TRAIN GROUPS        :", set(groups[train_idxs]))
    # print("TEST GROUPS         :", set(groups[test_idxs]))


Fold 0
ROC-AUC (LogisticRegression({'C': 1, 'class_weight': 'balanced', 'max_iter': 750, 'solver': 'lbfgs', 'warm_start': True})): 0.5
ROC-AUC (SVC({'C': 1, 'gamma': 'auto'})): 0.5
ROC-AUC (XGBoost({'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100, 'reg_alpha': 1})): 0.5293426418746111
Fold 1
ROC-AUC (LogisticRegression({'C': 1, 'class_weight': 'balanced', 'max_iter': 750, 'solver': 'lbfgs', 'warm_start': True})): 0.5
